# KITTI Object Detection — 5 Models
Запускать на Kaggle, GPU T4 x2 (device=0). P100 несовместим с torch 2.10.

In [ ]:
!git clone https://github.com/TYM4H/cv-kitti-detection.git
%cd cv-kitti-detection
!pip install ultralytics torchmetrics -q

In [ ]:
import torch
print(torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Конвертация датасета (2 мин, повторять после каждого restart)
!python src/dataset/convert_kitti.py \
    --images /kaggle/input/datasets/klemenko/kitti-dataset/data_object_image_2/training/image_2 \
    --labels /kaggle/input/datasets/klemenko/kitti-dataset/data_object_label_2/training/label_2 \
    --out    /kaggle/working/kitti_yolo

In [ ]:
# EDA: распределение классов
from src.utils.utils import plot_class_distribution
plot_class_distribution(
    labels_dir='/kaggle/working/kitti_yolo/labels/train',
    save_path='results/plots/class_distribution.png'
)

## Модель 1: YOLOv8n

In [ ]:
from src.training.train import run
r = run('yolov8n', kitti_yolo_dir='/kaggle/working/kitti_yolo')
print(r)

## Модель 2: YOLOv10n

In [ ]:
r = run('yolov10n', kitti_yolo_dir='/kaggle/working/kitti_yolo')
print(r)

## Модель 3: RT-DETR-L

In [ ]:
r = run('rtdetr', kitti_yolo_dir='/kaggle/working/kitti_yolo')
print(r)

## Модель 4: Faster R-CNN ResNet-50-FPN

In [ ]:
r = run('faster_rcnn', kitti_yolo_dir='/kaggle/working/kitti_yolo', batch=4, lr=0.005)
print(r)

## Модель 5: RetinaNet ResNet-50-FPN (Focal Loss)

In [ ]:
r = run('retinanet', kitti_yolo_dir='/kaggle/working/kitti_yolo', batch=4, lr=1e-4)
print(r)

## Сравнение всех моделей

In [ ]:
from src.utils.utils import print_results_table, plot_comparison
print_results_table('results/logs')
plot_comparison('results/logs', 'results/plots/comparison.png')

In [ ]:
# Сохранить результаты как артефакт
import shutil
shutil.make_archive('/kaggle/working/results_all', 'zip', 'results')